# Crash Course — 생리 신호 분석

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [crash-course/physiology-signals.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/crash-course/physiology-signals.md)

In [ ]:
# Colab 기본 런타임에 없는 패키지 설치
%pip install -q wfdb

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 표준 분석 흐름

실제 파일이 필요한 단계의 명령 형태(실행 예가 아니라 모양만):

In [ ]:
import wfdb
record = wfdb.rdrecord("<record 이름>", pn_dir="<PhysioNet 데이터베이스 경로>")
signal, fs = record.p_signal, record.fs

## 최소 실행 예시

### 예시 1 — 대역 통과 필터와 주파수 스펙트럼

10 Hz 리듬에 느린 drift, 60 Hz 전원 잡음, 백색 잡음을 섞은 가상 신호를 만들고 1–40 Hz 대역 통과 필터를 건다. 필터 전후의 대역별 파워를 비교한다.

In [ ]:
import numpy as np
from scipy import signal

fs = 250                                  # sampling rate (Hz)
t = np.arange(0, 20, 1 / fs)              # 20초
rng = np.random.default_rng(0)
x = (1.0 * np.sin(2 * np.pi * 10 * t)     # 관심 신호: 10 Hz 리듬
     + 2.0 * np.sin(2 * np.pi * 0.2 * t)  # 느린 기저선 흔들림 (drift)
     + 0.8 * np.sin(2 * np.pi * 60 * t)   # 전원 잡음 (60 Hz)
     + 0.5 * rng.standard_normal(t.size)) # 백색 잡음

# 1–40 Hz 대역 통과, 4차 Butterworth, 앞뒤로 두 번 걸어 위상 지연 없음
sos = signal.butter(4, [1, 40], btype="bandpass", fs=fs, output="sos")
y = signal.sosfiltfilt(sos, x)

def band_power(sig, lo, hi):
    f, pxx = signal.welch(sig, fs=fs, nperseg=fs * 4)   # 4초 창 → 0.25 Hz 해상도
    keep = (f >= lo) & (f < hi)
    return pxx[keep].sum() * (f[1] - f[0])

print(f"표본 수 {x.size}, Nyquist 주파수 {fs / 2} Hz")
print(f"{'대역':<14}{'필터 전':>10}{'필터 후':>10}")
for name, lo, hi in [("drift 0–1 Hz", 0, 1), ("신호 8–12 Hz", 8, 12), ("전원 58–62 Hz", 58, 62)]:
    print(f"{name:<14}{band_power(x, lo, hi):>12.4f}{band_power(y, lo, hi):>12.4f}")

f, pxx = signal.welch(y, fs=fs, nperseg=fs * 4)
print(f"필터 후 최대 peak 주파수: {f[np.argmax(pxx)]:.2f} Hz")
assert abs(f[np.argmax(pxx)] - 10) < 0.5

### 예시 2 — 참가자 단위 요약, 그리고 window를 독립 표본으로 셀 때 생기는 일

군당 6명, 한 사람당 window 20개인 가상 feature table을 만든다. **두 군 사이에 실제 차이는 없다.** 사람마다 고유한 수준이 있어서 한 사람의 window끼리는 서로 닮아 있다.

In [ ]:
import numpy as np, pandas as pd
from scipy import stats

def simulate(rng, n_per_group=6, n_windows=20, true_diff=0.0):
    """참가자마다 고유 수준(사람 간 SD 1.0) + window 잡음(SD 0.5)."""
    rows = []
    for g, group in enumerate(["control", "patient"]):
        for i in range(n_per_group):
            person_level = 10 + true_diff * g + rng.normal(0, 1.0)
            for w in range(n_windows):
                rows.append((f"{group[0].upper()}{i + 1:02d}", group, w,
                             person_level + rng.normal(0, 0.5)))
    return pd.DataFrame(rows, columns=["participant", "group", "window", "feature"])

rng = np.random.default_rng(1)
df = simulate(rng)                      # 실제 군 차이 없음
per_person = df.groupby(["group", "participant"], as_index=False)["feature"].mean()
print(f"window 행 수: {len(df)}, 참가자 수: {df['participant'].nunique()}")
print(per_person.groupby("group")["feature"].agg(["count", "mean", "std"]).round(2))

def p_window(d):   # 잘못된 방법: window를 독립 표본으로
    return stats.ttest_ind(*[d.loc[d.group == g, "feature"] for g in ["control", "patient"]],
                           equal_var=False).pvalue
def p_person(d):   # 올바른 방법: 참가자 평균으로 줄인 뒤 검정
    m = d.groupby(["group", "participant"])["feature"].mean()
    return stats.ttest_ind(m["control"], m["patient"], equal_var=False).pvalue

print(f"이 데이터의 p: window 단위 {p_window(df):.4f} (n=240), 참가자 단위 {p_person(df):.4f} (n=12)")

# 군 차이가 없는 데이터를 1,000번 만들어 p<0.05가 나오는 비율(거짓 양성률)
sims = [simulate(rng) for _ in range(1000)]
fp_w = np.mean([p_window(d) < 0.05 for d in sims])
fp_p = np.mean([p_person(d) < 0.05 for d in sims])
print(f"거짓 양성률: window 단위 {fp_w:.3f}, 참가자 단위 {fp_p:.3f} (기대값 0.05)")
assert fp_w > 0.3 and fp_p < 0.08